# 02 - Results: point accuracy, calibration, and what actually moves an interval

**Presentation layer only.** Every number and figure below is produced by
`src/evaluation.py`, `src/bootstrap.py` and `src/figures.py`, and read here from the
tables `run_all.py` wrote. This notebook defines no analysis logic of its own, and
re-running it cannot change a result. The command-line equivalent is:

```bash
python run_all.py --stage evaluate
python run_all.py --stage figures
```

Nothing here refits a model. Every table is a function of `data/processed/forecasts.csv`,
so no number below can move unless the backtest moves first.

## The questions this notebook answers

1. **Point accuracy.** Can the four models be separated on QLIKE, and with what
   uncertainty?
2. **Calibration.** Does a 95% predictive interval contain 95% of realised returns, and
   does a 99% VaR breach on 1% of days?
3. **Timing.** When a model does fail, do its failures cluster?
4. **Mechanism.** The frequentist and Bayesian models share a likelihood. What actually
   separates their intervals -- and how much of it is parameter uncertainty?

The fourth question is the project's subject, and it is the one where the obvious
comparison gives the wrong answer. See section 5.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from IPython.display import Image, display

from src import backtest as B

PROJECT_ROOT = Path.cwd().parent
PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "figures"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


def table(name: str) -> pd.DataFrame:
    return pd.read_csv(PROCESSED / f"{name}.csv")


HEADLINE = list(B.HEADLINE_MODELS)
print("headline models:", ", ".join(HEADLINE))
print("ablations, scored but not competitors:", B.BAYES_MEAN_MODEL, "and garch_mle_normal")


## 1. Which days each number is computed on

Two Bayesian refits failed their convergence diagnostics and produced no forecasts for
the 21 days each served, so `garch_bayes` has 2,092 of the 2,134 evaluation days. That
is a property of the model, not a gap to be filled, and decision **D28** says every table
must name the sample it used and carry its `n`:

- **per-model rows** use each model's own available days (`sample = "own days"`);
- **every pairwise comparison** uses that pair's intersection;
- the headline tables are additionally recomputed on the four-model common sample
  (`sample = "common sample"`), so a reader can check whether the missing 42 days matter.

They do not, at the resolution reported -- but that is a finding, not an assumption.


In [ ]:
coverage = table("eval_coverage")
coverage.groupby(["sample", "model"])["n"].max().unstack("sample")


## 2. Point accuracy

Mean QLIKE on the common sample, with a 95% stationary-block-bootstrap interval (mean
block length 20, 1,000 replications, fixed seed). The interval, not the ranking, is the
reportable object.

QLIKE is scored against the **scaled Parkinson proxy**, which is biased for the
close-to-close variance the models forecast; the comparison is within-proxy and the raw,
unscaled ranking is owed at Stage 6. Interval calibration below is scored against
observed returns and is unaffected by any of that.


In [ ]:
losses = table("eval_point_losses")
common = losses[(losses["sample"] == "common sample") & (losses["loss"] == "qlike")]
common.sort_values("mean")[["model", "n", "mean", "ci_lower", "ci_upper"]]


The governing plan predicted the four models would be hard to separate, with that
difficulty as the setup for the calibration act. Half of that is right. Both GARCH models
beat both baselines by a wide margin. What is hard to separate is the pair the research
question turns on.

Below, a **positive** `mean_differential` means `model_a` had the higher loss -- the worse
forecaster. `boot_excludes_zero` describes the interval, not the models: an interval
containing zero means this sample cannot tell them apart, which is a legitimate finding.

Read `differential_variance` before `dm_p`. The frequentist and Bayesian models share a
likelihood, so their differential can approach degeneracy, and DM is badly sized there.
The DM p-values are also unadjusted across eight comparisons, and DM's asymptotics assume
forecasts are not functions of estimated parameters -- which here they are. **The
bootstrap intervals carry the conclusions.**


In [ ]:
comparisons = table("eval_comparisons")
comparisons[
    [
        "model_a", "model_b", "n", "mean_differential", "differential_variance",
        "dm_statistic", "dm_p", "dm_lag_truncation",
        "boot_lower", "boot_upper", "boot_excludes_zero",
    ]
]


## 3. Where in the distribution each model is wrong

The probability integral transform: `u_t = F_t(r_t)`, each model's own predictive CDF
evaluated at the realised return, stored at forecast time. Under a correctly specified
predictive these are i.i.d. Uniform(0,1).

Finer than coverage at three fixed levels, because it shows *where* a model is
miscalibrated -- the difference between "the 95% interval is too narrow" and "the left
tail is too thin".

**The KS p-value is approximate**, and the table says so in a column rather than a
footnote: these predictive distributions have estimated parameters, re-estimated on a
rolling schedule, so the nominal p-value is optimistic. The histogram is the diagnostic;
the test is a summary of it.


In [ ]:
table("eval_pit").query("sample == 'own days'")[
    ["model", "n", "ks_stat", "ks_p", "p_value_is_approximate", "mean_pit"]
]


In [ ]:
display(Image(filename=str(FIGURES / "08_pit_histograms.png")))


## 4. Interval coverage

Empirical against nominal, scored on observed returns. `n_below` and `n_above` are
reported separately because a model can hit its nominal coverage overall while putting
its breaches almost entirely in one tail -- and for a risk model the lower tail is the
one that costs money.


In [ ]:
own = coverage.query("sample == 'own days' and model in @HEADLINE")
own.pivot(index="model", columns="nominal", values="empirical").reindex(HEADLINE)


In [ ]:
own[["model", "nominal", "empirical", "n", "n_below", "n_above", "mean_width"]]


In [ ]:
display(Image(filename=str(FIGURES / "09_coverage_vs_nominal.png")))


## 5. The 99% VaR, and the failure mode that actually appears

Three tests on the hit sequence: **Kupiec** on the number of breaches, **Christoffersen
independence** on whether they cluster, and the joint **conditional coverage** test.

The plan called independence the money test, on the argument that correct *average*
coverage can hide breaches that arrive together in a crisis. The result here is the
mirror image, and worth stating carefully.


In [ ]:
table("eval_var_backtests").query("sample == 'own days'")[
    [
        "model", "n", "breaches", "rate", "kupiec_p",
        "independence_p", "conditional_coverage_p",
    ]
]


In [ ]:
display(Image(filename=str(FIGURES / "10_var_hit_sequence.png")))


Every model fails Kupiec: far too many breaches, 37 against 21 expected even for the two
GARCH models. **Independence fires for none of them.** The models are wrong about the
*level* of tail risk and not about its *timing*.

One caution that belongs in the report. A non-rejection on a hit sequence of 37 events is
a failure to reject on a small sample, not a demonstration that breaches are well timed.
A first-order Markov test has little power there. "Independence did not reject" is the
honest sentence; "the breaches are independent" is not.


## 6. What actually moves the interval

This is the project's subject, and the obvious comparison gives the wrong answer.

`garch_mle` and `garch_bayes` share one likelihood, so the project's design note said any
difference between their intervals must be parameter uncertainty. That holds against a
plug-in at the **posterior mean**. It does not hold against a plug-in at the **MLE**,
which is what the forecast table contains, because the frozen priors move the point
estimate off the likelihood's maximum. Two causes, not one, and at the 99% level they
point in **opposite directions**.

`garch_bayes_mean` -- the plug-in predictive at the posterior mean, produced from the same
fits inside the same loop -- separates them:

| contrast | isolates |
|---|---|
| `garch_bayes` / `garch_bayes_mean` | parameter uncertainty, and nothing else |
| `garch_bayes_mean` / `garch_mle` | the priors, and nothing else |
| `garch_bayes` / `garch_mle` | the reported difference: both at once |

**No statement about parameter uncertainty may be sourced from the third row** (decision
D29). It is kept in the table so that it is visibly the composition of the other two.


In [ ]:
decomposition = table("eval_decomposition")
decomposition.query("regime == 'all'").pivot(
    index="nominal", columns="contrast", values="mean_width_ratio"
)


In [ ]:
decomposition.query("nominal == 0.99")[
    ["contrast", "regime", "n", "mean_width_ratio", "coverage_a", "coverage_b"]
]


In [ ]:
display(Image(filename=str(FIGURES / "11_interval_decomposition.png")))


Three things to read off those tables.

**Parameter uncertainty behaves exactly as the theory says, and is small.** It widens the
99% interval by 0.32% and slightly narrows the shoulders -- the crossover between 95% and
99% is the leptokurtosis of a scale mixture against the single distribution at its average
variance, not a bug. It is dwarfed by the priors, which narrow the 99% interval by 1.9%.

**The regime story belongs entirely to the priors.** At 99%, parameter uncertainty's
contribution is flat across calm, normal and stressed days (1.0036, 1.0031, 1.0027) while
the priors run 0.9927, 0.9766, 0.9683. "The Bayesian intervals behave differently in a
crisis" is true of the reported difference and false of parameter uncertainty.

**And it changes no coverage number at all.** `coverage_a` and `coverage_b` are identical
in the first contrast at every level and in every regime: nothing in 2,092 days of returns
lands in the 0.32% gap. That is the cleanest available statement of this project's central
measurement -- at these sample sizes, integrating over parameter uncertainty is not what
determines whether a risk model's intervals are calibrated.


## 7. What this stage does and does not establish

**Establishes.** Both GARCH models beat both baselines on QLIKE with intervals nowhere
near zero. Both GARCH-t models pass a PIT uniformity test that both baselines fail
decisively. All four under-cover at 99% and breach the 99% VaR far too often, and none of
them clusters those breaches. The frequentist-Bayesian interval difference is mostly the
priors, and parameter uncertainty moves no coverage number.

**Does not establish.** Nothing regime-conditional with error bars -- that is Stage 5, and
bootstrap CIs on per-regime coverage are on the never-cut list. Nothing about whether the
QLIKE ranking survives the raw, unscaled proxy (Stage 6, owed by D10). Nothing about
prior sensitivity (Stage 6, owed by D4). And nothing about power: the tests that did not
reject here were run on small hit sequences, and a test that cannot fire is not evidence.

**One asset, one horizon, two stress episodes.** Every conclusion above is conditional on
that, and the limitations section is on the never-cut list for exactly this reason.
